In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df=pd.read_csv("IMDB Dataset.csv")

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.shape

(50000, 2)

In [6]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [7]:
df.drop_duplicates(inplace=True)

In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

#text processing
1) convert to lowercase
2) remove urls,html,punctutation,stopwords
3) stemming
4) encoding
5) vectorization (converting words to numbers)

In [9]:
##converting to lowercase
df["review"]=df["review"].str.lower()



In [10]:
##removing url
import re

def remove_urls(text):
    # Check if text is a string (handles NaN / NoneType)
    if isinstance(text, str):
        return re.sub(r"http\S+", "", text)
    return text  # Return as-is if it's NaN/None

df["review"] = df["review"].apply(remove_urls)


In [11]:
#removing punctuations
def remove_punctuations(text):
    if isinstance(text, str):
        # Added the closing square bracket ']' here
        return re.sub(r"[^A-Za-z0-9\s]", "", text) 
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [12]:
#remvoing html tags
def remove_html(text):
    if isinstance(text,str):
        return re.sub(r"<.*?>","",text)
    return text
df["review"]=df["review"].apply(remove_html)

In [13]:
#removing the stopwords
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Tejas\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Tejas\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Tejas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


In [15]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def remove_stopwords(text):
    # Skip if the value is not a string (e.g., NaN / None)
    if not isinstance(text, str):
        return text
        
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words("english"))
    
    # Filter tokens and join them back into a string
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
    return " ".join(filtered_tokens)

df["review"] = df["review"].apply(remove_stopwords)

In [16]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


In [17]:
#stemming
from nltk.stem import PorterStemmer
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"]=df["review"].apply(stemming)

In [18]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product br br film techniqu unass...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


In [20]:
#encoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

df["sentiment"]=le.fit_transform(df["sentiment"])

In [21]:
y=df["sentiment"]

In [23]:
#vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)

x=tf.fit_transform(df["review"])
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4108209 stored elements and shape (49582, 5000)>

In [27]:

from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42
)





In [28]:
#converting sparse matrix to array
x_train=x_train.toarray()
x_test=x_test.toarray()


In [30]:
from torch.utils.data import DataLoader,TensorDataset

train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [31]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,batch_size=64,shuffle=True)


building RNN model
    

In [38]:
import torch.optim as optim
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()
        self.hidden_size=hidden_size
        self.num_layers=num_layers

        #rnn layer
        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
        #fully connected layer
        self.fc=nn.Linear(hidden_size,1)

    def forward(self,x):
                        # shape(num of layers,batch_size,hidden size)
        h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)
        out,_=self.rnn(x,h0)
        out=self.fc(out[:,-1,:])
        return out
        
        
        

In [39]:
input_size=x_train.shape[1]
model=RNN(input_size)
criterion=nn.BCELoss() #binary cross entropy loss
optimizer=optim.Adam(model.parameters())


training the model

In [41]:
epochs=10

for epoch in range(epochs):
    model.train()

    for xb,yb in train_loader:
        optimizer.zero_grad()
        xb=xb.unsqueeze(1) #add singleton direction
        outputs=model(xb)
        outputs=torch.sigmoid(outputs.squeeze()) #converting into probablity

        loss=criterion(outputs,yb)
        loss.backward()
        optimizer.step()

    print(f"epoch={epoch+1}/{epochs}and loss={loss.item()}")

epoch=1/10and loss=0.327338308095932


epoch=2/10and loss=0.2028312236070633


epoch=3/10and loss=0.2948009669780731


epoch=4/10and loss=0.2045956254005432


epoch=5/10and loss=0.12576182186603546


epoch=6/10and loss=0.2627331614494324


epoch=7/10and loss=0.20746268332004547


epoch=8/10and loss=0.19697953760623932


epoch=9/10and loss=0.20719915628433228


epoch=10/10and loss=0.08005969971418381


In [42]:
#evalution
model.eval()

with torch.no_grad():
    correct_vals=0
    total_vals=0
    for xb,yb in test_loader:
        xb=xb.unsqueeze(1)

        outputs=model(xb)
        predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()
        total_vals+=yb.size(0)
        correct_vals=(predicted==yb).sum().item()
    print(f"accuracy={correct_vals/total_vals*100}")
        
    

accuracy=0.5243521226177271
